# 프로젝트: 번역기 만들기

## Step 0. 라이브러리 확인



In [ ]:
import torch
import numpy
import matplotlib

print(torch.__version__)
print(numpy.__version__)
print(matplotlib.__version__)

2.11.0+cu128
2.0.2
3.10.0


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Step 1. 데이터 다운로드



In [ ]:
!mkdir data
%cd /content/data
!wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz
!gzip -d korean-english-park.train.tar.gz
!tar -xvf korean-english-park.train.tar

/content/data
--2026-07-10 07:11:53--  https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz [following]
--2026-07-10 07:11:54--  https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8718893 (8.3M) [application/octet-stream]
Saving to: ‘korean-english-park.train.tar.g

## Step 2. 데이터 정제 및 토큰화

### 2-1. 중복 제거

In [ ]:
data_dir = '/content/data'
kor_path = data_dir + "/korean-english-park.train.ko"
eng_path = data_dir + "/korean-english-park.train.en"

def clean_corpus(kor_path, eng_path):
    with open(kor_path, "r") as f: kor = f.read().splitlines()
    with open(eng_path, "r") as f: eng = f.read().splitlines()
    assert len(kor) == len(eng)

    cleaned_corpus = list(set(["\t".join([k, e]) for k, e in zip(kor, eng)]))

    return cleaned_corpus

cleaned_corpus = clean_corpus(kor_path, eng_path)

### 2-2. 정제 함수

In [ ]:
import re

def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = sentence.strip()
    return sentence

### 2-3. 토크나이저 생성

In [ ]:
!pip install sentencepiece

In [ ]:
def generate_tokenizer(corpus,
                        vocab_size,
                        lang="ko",
                        pad_id=0,
                        bos_id=1,
                        eos_id=2,
                        unk_id=3):
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus:
            f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=%s --model_prefix=%s --vocab_size=%d '
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'
        % (file, model, vocab_size, pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer


SRC_VOCAB_SIZE = TGT_VOCAB_SIZE = 20000

eng_corpus = []
kor_corpus = []

for pair in cleaned_corpus:
    k, e = pair.split("\t")
    kor_corpus.append(preprocess_sentence(k))
    eng_corpus.append(preprocess_sentence(e))

ko_tokenizer = generate_tokenizer(kor_corpus, SRC_VOCAB_SIZE, "ko")
en_tokenizer = generate_tokenizer(eng_corpus, TGT_VOCAB_SIZE, "en")
en_tokenizer.set_encode_extra_options("bos:eos")

True

### 2-4. 텐서 변환

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

src_corpus = []
tgt_corpus = []

assert len(kor_corpus) == len(eng_corpus)

for idx in tqdm(range(len(kor_corpus))):
    src_tokens = ko_tokenizer.encode_as_ids(kor_corpus[idx])
    tgt_tokens = en_tokenizer.encode_as_ids(eng_corpus[idx])

    if len(src_tokens) <= 50 and len(tgt_tokens) <= 50:
        src_corpus.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_corpus.append(torch.tensor(tgt_tokens, dtype=torch.long))

def pad_sequences(sequences, padding_value=0):
    return torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=padding_value)

enc_train = pad_sequences(src_corpus, padding_value=0)
dec_train = pad_sequences(tgt_corpus, padding_value=0)

print(enc_train.shape, dec_train.shape)

  0%|          | 0/78968 [00:00<?, ?it/s]

torch.Size([72107, 50]) torch.Size([72107, 50])


## Step 3. 모델 설계

### 3-1. Positional Encoding

In [ ]:
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table

### 3-2. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = K.shape[-1]
        QK = torch.matmul(Q, K.transpose(-2, -1))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        if mask is not None:
            scaled_qk = scaled_qk.masked_fill(mask == 1, float('-1e9'))

        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)

        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, d_model = x.shape
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask)

        out = self.combine_heads(out)
        out = self.linear(out)

        return out, attention_weights

### 3-3. Feed Forward

In [ ]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)

        return out

### 3-4. Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.dropout(out)
        out += residual

        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, enc_attn

### 3-5. Decoder Layer

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_mask, cross_mask):
        """
        Masked Multi-Head Attention (self-attention, causal + padding)
        """
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, self_mask)
        out = self.dropout(out)
        out += residual

        """
        Encoder-Decoder Multi-Head Attention (padding only)
        """
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, cross_mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn

### 3-6. Encoder / Decoder 스택

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)   # 추가

    def forward(self, x, mask):
        out = x
        enc_attns = []

        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)

        out = self.final_norm(out)   # 추가
        return out, enc_attns

In [ ]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)   # 추가

    def forward(self, x, enc_out, self_mask, cross_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []

        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](
                out, enc_out, self_mask, cross_mask
            )
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)

        out = self.final_norm(out)   # 추가
        return out, dec_attns, dec_enc_attns

### 3-7. Transformer 전체

In [ ]:
class Transformer(nn.Module):
    def __init__(self,
                 n_layers,
                 d_model,
                 n_heads,
                 d_ff,
                 src_vocab_size,
                 tgt_vocab_size,
                 pos_len,
                 dropout=0.2,
                 shared=True):

        super(Transformer, self).__init__()

        self.d_model = d_model
        self.shared = shared

        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        pos_table = positional_encoding(pos_len, d_model)
        self.register_buffer('pos_encoding', torch.tensor(pos_table, dtype=torch.float32))

        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        if shared:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        seq_len = x.shape[1]
        out = emb(x)

        out = out * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32, device=x.device))

        pos_enc = self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = out + pos_enc

        out = self.dropout(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, cross_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_in, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, dec_mask, cross_mask)

        logits = self.fc(dec_out)

        return logits, enc_attns, dec_attns, dec_enc_attns

### 3-8. 마스크 함수

In [ ]:
def generate_padding_mask(seq):
    """ 패딩된 부분(0)을 1로 표시 (1 = 마스킹 대상) """
    mask = (seq == 0).float()
    return mask[:, None, None, :]

def generate_causality_mask(tgt_len, src_len):
    """ 미래 정보를 참조하지 않도록 하는 Causal Mask (1 = 마스킹 대상) """
    mask = 1 - torch.tril(torch.ones(tgt_len, src_len))
    return mask.float()

def generate_masks(src, tgt):
    """ Encoder-Decoder에서 사용할 마스크 생성 """
    enc_mask = generate_padding_mask(src)          # Encoder self-attention
    dec_enc_mask = generate_padding_mask(src)      # Decoder-Encoder cross-attention (padding만)

    dec_padding_mask = generate_padding_mask(tgt)
    dec_causality_mask = generate_causality_mask(tgt.shape[1], tgt.shape[1]).to(tgt.device)
    dec_mask = torch.max(dec_padding_mask, dec_causality_mask)  # Decoder self-attention (causal + padding)

    return enc_mask, dec_enc_mask, dec_mask

## Step 4. 훈련하기

### 4-1. 모델 선언

In [ ]:
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=200,
    dropout=0.2,
    shared=True
)

### 4-2. Learning Rate Scheduler & Optimizer

In [ ]:
import math

class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super(LearningRateScheduler, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(1, self.last_epoch)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]


# 1. Optimizer를 먼저 선언합니다. (lr은 scheduler가 매 step마다 덮어쓰므로 초기값은 의미 없음)
optimizer = torch.optim.Adam(transformer.parameters(),
                              lr=0.0,
                              betas=(0.9, 0.98),
                              eps=1e-9)

# 2. 그 Optimizer를 감싸는 Scheduler를 선언합니다.
scheduler = LearningRateScheduler(optimizer, d_model=512, warmup_steps=4000)

### 4-3. Loss 함수



In [ ]:
loss_object = torch.nn.CrossEntropyLoss(reduction='none')

def loss_function(real, pred):
    # pred: (batch, seq_len, vocab_size) -> (batch * seq_len, vocab_size)
    # real: (batch, seq_len) -> (batch * seq_len,)
    vocab_size = pred.shape[-1]
    pred = pred.reshape(-1, vocab_size)
    real = real.reshape(-1)

    mask = (real != 0).float()
    loss_ = loss_object(pred, real)

    # Masking 되지 않은 입력의 개수로 Scaling
    loss_ = loss_ * mask

    return loss_.sum() / mask.sum()

### 4-4. train_step 함수

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer.to(device)

def train_step(src, tgt, model, optimizer, scheduler):
    src = src.to(device)
    tgt = tgt.to(device)
    gold = tgt[:, 1:]

    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt)

    optimizer.zero_grad()
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt, enc_mask, dec_enc_mask, dec_mask)
    loss = loss_function(gold, predictions[:, :-1])

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # 추가

    optimizer.step()
    scheduler.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

### 4-5. 번역 생성 / Attention 시각화 / 학습 루프

In [ ]:
import matplotlib.pyplot as plt

def evaluate(sentence, model, src_tokenizer, tgt_tokenizer):
    model.eval()
    sentence = preprocess_sentence(sentence)

    pieces = src_tokenizer.encode_as_pieces(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)

    _input = torch.tensor(tokens).unsqueeze(0).to(device)

    ids = []
    output = torch.tensor([tgt_tokenizer.bos_id()]).unsqueeze(0).to(device)

    with torch.no_grad():
        for i in range(dec_train.shape[-1]):
            enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(_input, output)

            predictions, enc_attns, dec_attns, dec_enc_attns = \
                model(_input, output, enc_padding_mask, combined_mask, dec_padding_mask)

            predicted_id = torch.argmax(torch.softmax(predictions, dim=-1)[0, -1]).item()

            if tgt_tokenizer.eos_id() == predicted_id:
                result = tgt_tokenizer.decode_ids(ids)
                model.train()
                return pieces, result, enc_attns, dec_attns, dec_enc_attns

            ids.append(predicted_id)
            output = torch.cat([output, torch.tensor([[predicted_id]]).to(device)], dim=-1)

    result = tgt_tokenizer.decode_ids(ids)
    model.train()
    return pieces, result, enc_attns, dec_attns, dec_enc_attns

In [ ]:
# attention 시각화
def visualize_attention(src, tgt, enc_attns, dec_attns, dec_enc_attns):
    def draw(data, ax, x="auto", y="auto"):
        import seaborn
        seaborn.heatmap(data.cpu(),
                        square=True,
                        vmin=0.0, vmax=1.0,
                        cbar=False, ax=ax,
                        xticklabels=x,
                        yticklabels=y)

    for layer in range(0, 2, 1):
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        print("Encoder Layer", layer + 1)
        for h in range(4):
            draw(enc_attns[layer][0, h, :len(src), :len(src)], axs[h], src, src)
        plt.show()

    for layer in range(0, 2, 1):
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        print("Decoder Self Layer", layer + 1)
        for h in range(4):
            draw(dec_attns[layer][0, h, :len(tgt), :len(tgt)], axs[h], tgt, tgt)
        plt.show()

        print("Decoder Src Layer", layer + 1)
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        for h in range(4):
            draw(dec_enc_attns[layer][0, h, :len(tgt), :len(src)], axs[h], src, tgt)
        plt.show()

In [ ]:
# 학습 시각화
def translate(sentence, model, src_tokenizer, tgt_tokenizer, plot_attention=False):
    pieces, result, enc_attns, dec_attns, dec_enc_attns = \
        evaluate(sentence, model, src_tokenizer, tgt_tokenizer)

    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

    if plot_attention:
        visualize_attention(pieces, result.split(), enc_attns, dec_attns, dec_enc_attns)

In [ ]:
# 학습

import random
from tqdm import tqdm

BATCH_SIZE = 64
EPOCHS = 20

examples = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

for epoch in range(EPOCHS):
    total_loss = 0

    idx_list = list(range(0, enc_train.shape[0], BATCH_SIZE))
    random.shuffle(idx_list)
    t = tqdm(idx_list)

    for (batch, idx) in enumerate(t):
        batch_loss, enc_attns, dec_attns, dec_enc_attns = \
            train_step(enc_train[idx:idx + BATCH_SIZE],
                       dec_train[idx:idx + BATCH_SIZE],
                       transformer,
                       optimizer,
                       scheduler)

        total_loss += batch_loss.item()   # 그래프 누적 방지를 위해 item()으로 변환

        t.set_description('Epoch %2d' % (epoch + 1))
        t.set_postfix({'Loss': '%.4f' % (total_loss / (batch + 1))})

    for example in examples:
        translate(example, transformer, ko_tokenizer, en_tokenizer)

Epoch  1: 100%|██████████| 1127/1127 [03:09<00:00,  5.94it/s, Loss=118.2412]


Input: 오바마는 대통령이다.
Predicted translation: the
Input: 시민들은 도시 속에 산다.
Predicted translation: the
Input: 커피는 필요 없다.
Predicted translation: the
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the


Epoch  2: 100%|██████████| 1127/1127 [03:10<00:00,  5.91it/s, Loss=12.3430]


Input: 오바마는 대통령이다.
Predicted translation: the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the ,
Input: 시민들은 도시 속에 산다.
Predicted translation: the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the ,
Input: 커피는 필요 없다.
Predicted translation: the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the ,
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the , the ,


Epoch  3: 100%|██████████| 1127/1127 [03:10<00:00,  5.90it/s, Loss=7.7071]


...

Input: 오바마는 대통령이다.
Predicted translation: obama is the first time for the democratic presidential nomination .
Input: 시민들은 도시 속에 산다.
Predicted translation: the ap s largest was a smaller .
Input: 커피는 필요 없다.
Predicted translation: i think that i think .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the death toll were killed in the southern city of the southern city of the deaths .


Epoch 19: 100%|██████████| 1127/1127 [03:11<00:00,  5.87it/s, Loss=4.6858]


Input: 오바마는 대통령이다.
Predicted translation: the president elect is the first time in the presidential election .
Input: 시민들은 도시 속에 산다.
Predicted translation: the death toll in the city .
Input: 커피는 필요 없다.
Predicted translation: i think that i think that i think that i think that i can t be .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the death toll in the death toll in the deaths were killed in the area .


Epoch 20: 100%|██████████| 1127/1127 [03:11<00:00,  5.88it/s, Loss=4.6480]


Input: 오바마는 대통령이다.
Predicted translation: the president elect barack obama is the first time .
Input: 시민들은 도시 속에 산다.
Predicted translation: the number of the most of the city of the city of the city of the city of the city of the city of the city of the city of the city of the city of the city of the city of the city of the city of the city
Input: 커피는 필요 없다.
Predicted translation: it s not a lot of the first time .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the death toll were killed in the area .


loss도 너무 크고, 번역 결과도 너무 이상해서 코드 수정 필요  
Encoder, Decoder 클래스에 최종 LayerNorm을 추가

코드 수정을 했지만 생각보다 번역결과가 별로이다.

In [ ]:
loss_object = torch.nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1)

In [ ]:
# 직전에 나온 n-gram과 동일한 토큰이 반복되면 억제하는 evaluate 함수
# 기존 evaluate()는 그대로 두고, 새 함수명으로 덮어써서 이후 translate가 이걸 쓰게 합니다.

def evaluate(sentence, model, src_tokenizer, tgt_tokenizer,
             no_repeat_ngram_size=3, repeat_penalty=5.0):
    model.eval()
    sentence = preprocess_sentence(sentence)

    pieces = src_tokenizer.encode_as_pieces(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)

    _input = torch.tensor(tokens).unsqueeze(0).to(device)

    ids = []
    output = torch.tensor([tgt_tokenizer.bos_id()]).unsqueeze(0).to(device)

    with torch.no_grad():
        for i in range(dec_train.shape[-1]):
            enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(_input, output)

            predictions, enc_attns, dec_attns, dec_enc_attns = \
                model(_input, output, enc_padding_mask, combined_mask, dec_padding_mask)

            logits = predictions[0, -1].clone()

            # --- 반복 억제: 이미 나온 (n-1)-gram 뒤에 올 다음 토큰이
            # 과거에 등장했던 토큰이면 확률을 낮춤 ---
            if no_repeat_ngram_size > 0 and len(ids) >= no_repeat_ngram_size - 1:
                prefix = tuple(ids[-(no_repeat_ngram_size - 1):]) if no_repeat_ngram_size > 1 else tuple()
                banned_tokens = set()
                for idx in range(len(ids) - (no_repeat_ngram_size - 1)):
                    if tuple(ids[idx: idx + (no_repeat_ngram_size - 1)]) == prefix:
                        if idx + (no_repeat_ngram_size - 1) < len(ids):
                            banned_tokens.add(ids[idx + (no_repeat_ngram_size - 1)])
                for t in banned_tokens:
                    logits[t] -= repeat_penalty

            predicted_id = torch.argmax(torch.softmax(logits, dim=-1)).item()

            if tgt_tokenizer.eos_id() == predicted_id:
                result = tgt_tokenizer.decode_ids(ids)
                model.train()
                return pieces, result, enc_attns, dec_attns, dec_enc_attns

            ids.append(predicted_id)
            output = torch.cat([output, torch.tensor([[predicted_id]]).to(device)], dim=-1)

    result = tgt_tokenizer.decode_ids(ids)
    model.train()
    return pieces, result, enc_attns, dec_attns, dec_enc_attns

In [ ]:
examples = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

for example in examples:
    translate(example, transformer, ko_tokenizer, en_tokenizer)

Input: 오바마는 대통령이다.
Predicted translation: the president elect barack obama is the first time .
Input: 시민들은 도시 속에 산다.
Predicted translation: the number of the most of the city of the world s largest .
Input: 커피는 필요 없다.
Predicted translation: it s not a lot of the first time .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the death toll were killed in the area .


데이터 정제 강화 + 검증셋 분리

In [ ]:
# 오정렬 의심 쌍 제거 + train/valid 분리
import random

filtered_pairs = []
for k_sent, e_sent, s_toks, t_toks in zip(kor_corpus, eng_corpus,
                                          [ko_tokenizer.encode_as_ids(s) for s in kor_corpus],
                                          [en_tokenizer.encode_as_ids(s) for s in eng_corpus]):
    ls, lt = len(s_toks), len(t_toks)
    if ls < 2 or lt < 4:                 # 너무 짧은 문장 제거 (en은 bos/eos 포함)
        continue
    if ls > 50 or lt > 50:               # 기존 길이 제한 유지
        continue
    if lt / ls > 3.0 or ls / lt > 3.0:   # 길이 비율 3배 초과 → 오정렬 의심
        continue
    filtered_pairs.append((torch.tensor(s_toks, dtype=torch.long),
                           torch.tensor(t_toks, dtype=torch.long)))

random.seed(42)
random.shuffle(filtered_pairs)

n_valid = 1000
valid_pairs = filtered_pairs[:n_valid]
train_pairs = filtered_pairs[n_valid:]

enc_train = pad_sequences([p[0] for p in train_pairs])
dec_train = pad_sequences([p[1] for p in train_pairs])
enc_valid = pad_sequences([p[0] for p in valid_pairs])
dec_valid = pad_sequences([p[1] for p in valid_pairs])

print(f"필터링 후: {len(filtered_pairs)}쌍 (train {len(train_pairs)} / valid {n_valid})")
print(enc_train.shape, dec_train.shape)

필터링 후: 70730쌍 (train 69730 / valid 1000)
torch.Size([69730, 50]) torch.Size([69730, 50])


모델 재선언 + Xavier 초기화 + 새 설정

In [ ]:
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=200,
    dropout=0.1,          # 0.2 -> 0.1
    shared=True
)

# Xavier 초기화 (임베딩 포함 — 초기 logit 폭발 방지)
for p in transformer.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

transformer.to(device)

optimizer = torch.optim.Adam(transformer.parameters(),
                              lr=0.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = LearningRateScheduler(optimizer, d_model=512, warmup_steps=4000)

# Label smoothing 적용된 loss (재학습이므로 처음부터 효과 적용됨)
loss_object = torch.nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1)

BLEU 평가 함수

In [ ]:
!pip install sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00


In [ ]:
import sacrebleu

def compute_bleu(model, n_samples=200):
    """검증셋 일부에 대해 greedy 디코딩 후 BLEU 측정"""
    model.eval()
    hyps, refs = [], []

    with torch.no_grad():
        for i in range(n_samples):
            src = enc_valid[i:i+1].to(device)
            src = src[:, :int((src != 0).sum())]          # 패딩 제거

            output = torch.tensor([[en_tokenizer.bos_id()]]).to(device)
            ids = []
            for _ in range(60):
                m1, m2, m3 = generate_masks(src, output)
                pred, _, _, _ = model(src, output, m1, m2, m3)
                nid = torch.argmax(pred[0, -1]).item()
                if nid == en_tokenizer.eos_id():
                    break
                ids.append(nid)
                output = torch.cat([output, torch.tensor([[nid]]).to(device)], dim=-1)

            hyps.append(en_tokenizer.decode_ids(ids))
            ref_ids = [t for t in dec_valid[i].tolist()
                       if t not in (0, en_tokenizer.bos_id(), en_tokenizer.eos_id())]
            refs.append(en_tokenizer.decode_ids(ref_ids))

    model.train()
    return sacrebleu.corpus_bleu(hyps, [refs]).score

# 참고: 기존 학습된 모델이 아직 메모리에 있다면 여기서 먼저
# print(compute_bleu(transformer)) 로 "기준 점수"를 재보고 셀 B를 실행하는 것도 좋습니다.

재학습 (epoch별 Loss/BLEU 기록 포함)

In [ ]:
BATCH_SIZE = 64
EPOCHS = 20

epoch_losses = []
bleu_history = []

for epoch in range(EPOCHS):
    total_loss = 0
    idx_list = list(range(0, enc_train.shape[0], BATCH_SIZE))
    random.shuffle(idx_list)
    t = tqdm(idx_list)

    for (batch, idx) in enumerate(t):
        batch_loss, *_ = train_step(enc_train[idx:idx+BATCH_SIZE],
                                    dec_train[idx:idx+BATCH_SIZE],
                                    transformer, optimizer, scheduler)
        total_loss += batch_loss.item()
        t.set_description('Epoch %2d' % (epoch + 1))
        t.set_postfix({'Loss': '%.4f' % (total_loss / (batch + 1))})

    epoch_losses.append(total_loss / len(idx_list))

    bleu = compute_bleu(transformer)
    bleu_history.append(bleu)
    print(f"Epoch {epoch+1} | Loss {epoch_losses[-1]:.4f} | Valid BLEU {bleu:.2f}")

    for example in examples:
        translate(example, transformer, ko_tokenizer, en_tokenizer)

Epoch  1: 100%|██████████| 1090/1090 [03:07<00:00,  5.83it/s, Loss=7.0680]


Epoch 1 | Loss 7.0680 | Valid BLEU 1.32
Input: 오바마는 대통령이다.
Predicted translation: obama s  ⁇ 
Input: 시민들은 도시 속에 산다.
Predicted translation: the new york such .
Input: 커피는 필요 없다.
Predicted translation: the new york s  ⁇ 
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the u . s . s s .


Epoch  2: 100%|██████████| 1090/1090 [03:07<00:00,  5.81it/s, Loss=5.6205]


Epoch 2 | Loss 5.6205 | Valid BLEU 2.08
Input: 오바마는 대통령이다.
Predicted translation: obama is agreed to win .
Input: 시민들은 도시 속에 산다.
Predicted translation: the people were in the world .
Input: 커피는 필요 없다.
Predicted translation: it s not a lot of the problem .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: the death toll was killed .


Epoch  3: 100%|██████████| 1090/1090 [03:07<00:00,  5.82it/s, Loss=5.0734]


...

Epoch 14 | Loss 3.0719 | Valid BLEU 6.39
Input: 오바마는 대통령이다.
Predicted translation: obama is the presumptive nominee .
Input: 시민들은 도시 속에 산다.
Predicted translation: they are in the city .
Input: 커피는 필요 없다.
Predicted translation: coffee is not a good cup .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: seven deaths were reported .


Epoch 15: 100%|██████████| 1090/1090 [03:07<00:00,  5.81it/s, Loss=2.9907]


Epoch 15 | Loss 2.9907 | Valid BLEU 6.53
Input: 오바마는 대통령이다.
Predicted translation: obama is the presumptive presidential nominee .
Input: 시민들은 도시 속에 산다.
Predicted translation: they are in the city s san .
Input: 커피는 필요 없다.
Predicted translation: coffee is no alternative .
Input: 일곱 명의 사망자가 발생했다.
Predicted translation: seven deaths were reported .


Epoch 16:  42%|████▏     | 463/1090 [01:19<01:48,  5.80it/s, Loss=2.8395]


KeyboardInterrupt: 

epoch 10부터 loss는 떨어지지만 bleu가 정체되고 오히려 감소하여 학습 종료. epoch 10이 가장 좋은 번역 결과를 보여준다

![](10.png)

회고  
처음 학습을 돌렸을 때 Epoch 1 Loss가 6027이 나왔다. 정상 범위(ln(vocab_size)≈9.9)와 비교하면 명백히 잘못된 값이었는데, 원인은 Pre-LN 구조에서 Encoder/Decoder 마지막에 LayerNorm이 빠져 있었기 때문이었다.  
Xavier 초기화: 기존엔 PyTorch 기본 초기화(N(0,1))를 그대로 썼는데, 여기에 sqrt(d_model) 스케일링과 임베딩 가중치 공유가 겹치면서 초기 logit이 과도하게 커지는 게 원인이었다. Xavier로 바꾸니 Epoch 1부터 Loss가 정상 범위(7대)에서 출발했다.
Label Smoothing (0.1): 원 논문에서 실제로 쓰인 기법인데 처음엔 빠뜨렸었다. 모델이 정답 토큰에 과도하게 확신하는 걸 막아주면서, 반복 루프 완화에도 도움이 됐다.
데이터 정제 강화: 길이 비율이 지나치게 차이 나는 문장 쌍(오정렬 의심)을 걸러내고, 검증셋을 따로 분리했다.
재학습을 시작하면서 검증셋 기반 BLEU 측정 함수를 추가했는데, 이게 있고 나서야 "이번 레시피가 이전보다 나은지", "몇 epoch가 최적점인지"를 감이 아니라 숫자로 판단할 수 있었다. 실제로 Epoch 12에서 BLEU가 6.76으로 최고점을 찍은 뒤 이후 epoch에서 계속 그 아래에 머무는 걸 보고 과적합 시점을 잡을 수 있었다.